# Week 5 — Second-Order and Quasi-Newton Methods
**80-minute computer seminar.** Implement Newton and the BFGS update yourself. Use SciPy implementations only in the comparison section.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.sparse.linalg import LinearOperator, cg
rng = np.random.default_rng(42)
plt.rcParams.update({'figure.figsize': (6, 4), 'axes.grid': True})

## 1. Newton's method
Implement full-step Newton first. Solve $Hp=-g$ with a linear solver. Never form an explicit matrix inverse.

In [ ]:
def newton(f, grad, hess, x0, tol=1e-8, max_iter=100):
    x = np.asarray(x0, dtype=float).copy()
    history = {'x': [x.copy()], 'f': [float(f(x))], 'grad_norm': [],
               'step': [], 'f_calls': 1, 'g_calls': 0, 'h_calls': 0}
    for _ in range(max_iter):
        g = np.asarray(grad(x), float); history['g_calls'] += 1
        H = np.asarray(hess(x), float); history['h_calls'] += 1
        gn = np.linalg.norm(g); history['grad_norm'].append(gn)
        if gn <= tol:
            break
        # TODO: solve H p = -g, update x, and record the step and objective.
        raise NotImplementedError
    return x, history

### Positive definite quadratic
Newton should reach the solution in one update up to floating-point error.

In [ ]:
Q = np.array([[4.0, 1.0], [1.0, 2.0]])
b = np.array([1.0, -1.0])
f_quad = lambda x: 0.5*x@Q@x - b@x
g_quad = lambda x: Q@x - b
H_quad = lambda x: Q
x_star = np.linalg.solve(Q, b)
# TODO: run newton from several initial points and verify one-step convergence.


### Rosenbrock function
Full Newton no longer has a one-step guarantee. Compare objective and gradient-norm histories.

In [ ]:
def rosenbrock(x):
    a, b = x
    return 100*(b-a*a)**2 + (1-a)**2

def rosen_grad(x):
    a, b = x
    return np.array([-400*a*(b-a*a)-2*(1-a), 200*(b-a*a)])

def rosen_hess(x):
    a, b = x
    return np.array([[1200*a*a-400*b+2, -400*a], [-400*a, 200.]])

# TODO: run full Newton from two initial points and inspect failures or slow starts.


## 2. An indefinite Hessian
At $(0,0.5)$, the Newton direction for the function below is an ascent direction.

In [ ]:
def saddle_quartic(x):
    a, b = x
    return 0.5*a*a - b*b + 0.25*b**4

def saddle_grad(x):
    a, b = x
    return np.array([a, -2*b+b**3])

def saddle_hess(x):
    _, b = x
    return np.diag([1.0, -2+3*b*b])

x_bad = np.array([0.0, 0.5])
g_bad, H_bad = saddle_grad(x_bad), saddle_hess(x_bad)
p_bad = np.linalg.solve(H_bad, -g_bad)
print('g =', g_bad, 'eigenvalues =', np.linalg.eigvalsh(H_bad))
print('Newton direction =', p_bad, 'g.T p =', g_bad @ p_bad)
# TODO: compare f(x+p), f(x), and a short gradient step.


## 3. Armijo backtracking and Hessian regularization
Increase $\lambda$ until $H+\lambda I$ is positive definite, then apply line search. Record both accepted $\alpha$ and $\lambda$.

In [ ]:
def armijo(f, x, g, p, alpha0=1.0, beta=0.5, c=1e-4, max_shrinks=60):
    if g @ p >= 0:
        raise ValueError('Armijo requires a descent direction')
    fx, alpha = float(f(x)), float(alpha0)
    for j in range(max_shrinks + 1):
        trial = float(f(x + alpha*p))
        if np.isfinite(trial) and trial <= fx + c*alpha*(g@p):
            return alpha, j
        alpha *= beta
    raise RuntimeError('line search failed')

def regularized_newton(f, grad, hess, x0, tol=1e-8, max_iter=200,
                       lambda0=1e-8, growth=10.0):
    x = np.asarray(x0, float).copy()
    history = {'x': [x.copy()], 'f': [float(f(x))], 'grad_norm': [],
               'alpha': [], 'lambda': [], 'f_calls': 1, 'g_calls': 0, 'h_calls': 0}
    for _ in range(max_iter):
        g, H = grad(x), hess(x)
        history['g_calls'] += 1; history['h_calls'] += 1
        history['grad_norm'].append(np.linalg.norm(g))
        if np.linalg.norm(g) <= tol:
            break
        lam = lambda0
        # TODO: increase lam until Cholesky succeeds and g.T@p < 0.
        # TODO: call armijo, update x, and append all history fields.
        raise NotImplementedError
    return x, history

### Visual comparison
Plot full Newton and regularized Newton on identical contours for the saddle-quartic and Rosenbrock functions.

In [ ]:
def contour_path(f, paths, xlim, ylim, levels=30):
    xs, ys = np.linspace(*xlim, 350), np.linspace(*ylim, 350)
    X, Y = np.meshgrid(xs, ys)
    Z = np.empty_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i,j] = f(np.array([X[i,j], Y[i,j]]))
    fig, ax = plt.subplots()
    ax.contour(X, Y, Z, levels=levels, cmap='Blues')
    for name, path in paths.items():
        path = np.asarray(path)
        ax.plot(path[:,0], path[:,1], 'o-', ms=3, label=name)
    ax.legend(); ax.set(xlabel='$x_1$', ylabel='$x_2$')
    return fig, ax
# TODO: create the path comparison.


## 4. Inverse BFGS update
Check curvature before updating. The numerical tests below should confirm symmetry, the secant equation, and positive eigenvalues.

In [ ]:
def inverse_bfgs_update(G, s, y, curvature_tol=1e-12):
    G, s, y = np.asarray(G, float), np.asarray(s, float), np.asarray(y, float)
    ys = y @ s
    if ys <= curvature_tol*np.linalg.norm(y)*np.linalg.norm(s):
        return G.copy(), False
    # TODO: implement (I-rho*s*y.T) G (I-rho*y*s.T) + rho*s*s.T
    raise NotImplementedError

G = np.eye(3)
s = np.array([1.0, -0.5, 0.3])
y = np.array([2.0, -0.2, 0.5])
# TODO: update, then print symmetry error, secant residual, and eigenvalues.


## 5. Regularized logistic regression
Implement stable loss, gradient, Hessian, and Hessian-vector product. Do not regularize an intercept if you add one.

In [ ]:
n, p = 3000, 20
X = rng.normal(size=(n, p)) * np.geomspace(1.0, 20.0, p)
w_true = rng.normal(size=p) / np.geomspace(1.0, 20.0, p)
scores = X @ w_true
prob = np.where(scores >= 0, 1/(1+np.exp(-scores)), np.exp(scores)/(1+np.exp(scores)))
y = rng.binomial(1, prob).astype(float)
reg = 1e-2

def sigmoid(z):
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1/(1+np.exp(-z[pos]))
    ez = np.exp(z[~pos]); out[~pos] = ez/(1+ez)
    return out

def logistic_loss(w):
    z = X @ w
    return np.mean(np.logaddexp(0.0, z)-y*z) + 0.5*reg*(w@w)

def logistic_grad(w):
    z = X @ w
    return X.T @ (sigmoid(z)-y) / n + reg*w

def logistic_hess(w):
    pr = sigmoid(X @ w); d = pr*(1-pr)
    return X.T @ (d[:,None]*X) / n + reg*np.eye(p)

def logistic_hvp(w, v):
    # TODO: compute X.T @ (d * (X @ v)) / n + reg*v without forming H.
    raise NotImplementedError


### Derivative and HVP checks
Use central differences for the gradient and a finite difference of gradients for $Hv$. Sweep several step sizes to distinguish truncation from round-off error.

In [ ]:
w_test = rng.normal(scale=0.05, size=p)
v_test = rng.normal(size=p); v_test /= np.linalg.norm(v_test)
# TODO: compare logistic_hvp(w_test, v_test) with logistic_hess(w_test) @ v_test.
# TODO: compare with [grad(w+eps*v)-grad(w-eps*v)]/(2*eps).


## 6. GD, Newton, BFGS, and L-BFGS benchmark
Use the same initial point and stopping tolerance. Count evaluations explicitly because SciPy's iteration count alone is not a fair work measure.

In [ ]:
class Counter:
    def __init__(self, fun): self.fun, self.calls = fun, 0
    def __call__(self, x):
        self.calls += 1
        return self.fun(x)

def timed_run(name, solve, repeats=3):
    times, results = [], []
    for _ in range(repeats):
        t0 = time.perf_counter(); result = solve(); times.append(time.perf_counter()-t0)
        results.append(result)
    return {'method': name, 'time_median': float(np.median(times)), 'result': results[-1]}

w0 = np.zeros(p)
# TODO: add your GD and regularized Newton runs.
# TODO: use minimize(..., method='BFGS') and method='L-BFGS-B'.
# TODO: collect objective, gradient norm, iterations, calls, and median time.


## 7. Box constraints with L-BFGS-B
Solve the logistic problem with $-1\leq w_j\leq1$. Report active bounds and a projected-gradient norm.

In [ ]:
bounds = [(-1.0, 1.0)] * p
# TODO: run L-BFGS-B. Count coordinates within 1e-7 of a bound.
# TODO: compute a projected gradient by zeroing components blocked by an active bound.


## 8. Matrix-free Newton-CG
Create a `LinearOperator` whose `matvec` calls `logistic_hvp`. Solve $Hp=-g$ with CG and compare with an explicit dense solve.

In [ ]:
def newton_cg_direction(w, rtol=1e-8, maxiter=None):
    g = logistic_grad(w)
    calls = {'hvp': 0}
    def matvec(v):
        calls['hvp'] += 1
        return logistic_hvp(w, v)
    op = LinearOperator((p, p), matvec=matvec, dtype=float)
    direction, info = cg(op, -g, rtol=rtol, atol=0.0, maxiter=maxiter)
    return direction, info, calls['hvp']

# TODO: compare direction accuracy, memory, and time with np.linalg.solve(H, -g).


## 9. Total-cost experiment
Repeat the benchmark while increasing $p$. Plot time per iteration and total time. Identify where fewer expensive Newton steps beat many cheap GD steps, and where dense Hessian storage becomes the limiting factor.

In [ ]:
# TODO: run a controlled size sweep. Keep the stopping rule and data-generation process fixed.
# Suggested columns: p, method, iterations, f_calls, g_calls, h_or_hvp_calls, time, final_grad_norm.


## Reflection
Explain: (1) why an indefinite Newton step can increase the objective; (2) why line search does not fix a non-descent direction; (3) how BFGS obtains curvature without Hessians; (4) when HVP-based Newton-CG has lower total cost.